# Лабораторная работа 6.1
## Транспортная задача

Реализация метода потенциалов для решения транспортной задачи минимизации стоимости перевозок.

In [1]:
import numpy as np
from collections import defaultdict, deque


def northwest_corner_method(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    m, n = len(a), len(b)
    
    x = np.zeros((m, n))
    B = set()
    
    a_copy = a.copy()
    b_copy = b.copy()
    i, j = 0, 0
    
    while i < m and j < n:
        B.add((i, j))
        x[i, j] = min(a_copy[i], b_copy[j])
        a_copy[i] -= x[i, j]
        b_copy[j] -= x[i, j]
        
        if a_copy[i] < 1e-9:
            i += 1
        else:
            j += 1
    
    return x, B


def compute_potentials(c, B):
    c = np.array(c, dtype=float)
    m, n = c.shape
    
    u = np.full(m, np.nan)
    v = np.full(n, np.nan)
    u[0] = 0
    
    adj_i = defaultdict(list)
    adj_j = defaultdict(list)
    for i, j in B:
        adj_i[i].append(j)
        adj_j[j].append(i)
    
    queue = deque([('i', 0)])
    visited = set()
    visited.add(('i', 0))
    
    while queue:
        node_type, node_idx = queue.popleft()

        if node_type == 'i':
            for j in adj_i[node_idx]:
                if np.isnan(v[j]):
                    v[j] = c[node_idx, j] - u[node_idx]
                    queue.append(('j', j))

        else:  # node_type == 'j'
            for i in adj_j[node_idx]:
                if np.isnan(u[i]):
                    u[i] = c[i, node_idx] - v[node_idx]
                    queue.append(('i', i))
    return u, v


def find_cycle_in_basis(B, i_new, j_new):
    B_temp = B | {(i_new, j_new)}
    
    adj_i = defaultdict(list)
    adj_j = defaultdict(list)
    for i, j in B_temp:
        adj_i[i].append(j)
        adj_j[j].append(i)
    
    stack = [(i_new, j_new, [(i_new, j_new)], 'i')]
    
    while stack:
        i, j, path, last_type = stack.pop()
        
        if last_type == 'i':
            for j_next in sorted(adj_i[i]):
                if j_next == j:
                    continue
                pos = (i, j_next)
                if pos == (i_new, j_new) and len(path) > 1:
                    return path
                if pos not in path:
                    stack.append((i, j_next, path + [pos], 'j'))
        else:
            for i_next in sorted(adj_j[j]):
                if i_next == i:
                    continue
                pos = (i_next, j)
                if pos == (i_new, j_new) and len(path) > 1:
                    return path
                if pos not in path:
                    stack.append((i_next, j, path + [pos], 'i'))
    
    raise RuntimeError(f"Цикл не найден для позиции ({i_new}, {j_new})")


def solve_transportation_problem(c, a, b, max_iter=1000, eps=1e-9):
    c = np.array(c, dtype=float)
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    m, n = c.shape
    
    if abs(sum(a) - sum(b)) > eps:
        raise ValueError("Задача не сбалансирована: сумма поставок не равна сумме заявок")
    
    x, B = northwest_corner_method(a, b)
    history = []
    
    for iteration in range(1, max_iter + 1):
        u, v = compute_potentials(c, B)
        
        min_delta = 0
        best_pos = None
        
        for i in range(m):
            for j in range(n):
                if (i, j) not in B:
                    delta = c[i, j] - u[i] - v[j]
                    if delta < min_delta - eps:
                        min_delta = delta
                        best_pos = (i, j)
        
        if best_pos is None:
            cost = float(np.sum(c * x))
            return "optimal", x, cost, B, history
        
        i_new, j_new = best_pos
        cycle = find_cycle_in_basis(B, i_new, j_new)
        
        signs = {}
        for idx, pos in enumerate(cycle):
            signs[pos] = '+' if idx % 2 == 0 else '-'
        
        theta = float('inf')
        exit_pos = None
        
        for (i, j), sign in signs.items():
            if sign == '-' and x[i, j] > eps:
                if x[i, j] < theta - eps:
                    theta = x[i, j]
                    exit_pos = (i, j)
        
        if exit_pos is None or theta == float('inf'):
            theta = 0
            for pos in sorted(signs.keys()):
                if signs[pos] == '-':
                    exit_pos = pos
                    break
        
        for (i, j), sign in signs.items():
            if sign == '+':
                x[i, j] += theta
            elif sign == '-':
                x[i, j] -= theta
        
        B.add(best_pos)
        B.discard(exit_pos)
        x[np.abs(x) < eps] = 0
        
        history.append({
            "iter": iteration,
            "B": B.copy(),
            "x": x.copy(),
            "u": u.copy(),
            "v": v.copy(),
            "new_pos": best_pos,
            "exit_pos": exit_pos,
            "theta": theta
        })
    
    raise RuntimeError(f"Превышено максимальное число итераций ({max_iter})")

In [2]:
c = np.array([
    [8, 4, 1],
    [8, 4, 3],
    [9, 7, 5]
])
a = np.array([100, 300, 300])
b = np.array([300, 200, 200])

print("Пример 4: Начальный базисный план методом северо-западного угла")
print("Матрица стоимостей c:")
print(c)
print("\nПоставки a:", a)
print("Заявки b:", b)

status, x_opt, cost, B_opt, history = solve_transportation_problem(c, a, b)

print(f"\nСтатус: {status}")
print(f"Минимальная стоимость: {cost}")
print(f"\nОптимальный план перевозок:")
print(x_opt)
print(f"\nОптимальные базисные позиции (0-based):")
print(sorted(B_opt))
print(f"Оптимальные базисные позиции (1-based):")
print(sorted([(i+1, j+1) for i, j in B_opt]))

Пример 4: Начальный базисный план методом северо-западного угла
Матрица стоимостей c:
[[8 4 1]
 [8 4 3]
 [9 7 5]]

Поставки a: [100 300 300]
Заявки b: [300 200 200]

Статус: optimal
Минимальная стоимость: 3900.0

Оптимальный план перевозок:
[[  0.   0. 100.]
 [  0. 200. 100.]
 [300.   0.   0.]]

Оптимальные базисные позиции (0-based):
[(0, 2), (1, 1), (1, 2), (2, 0), (2, 2)]
Оптимальные базисные позиции (1-based):
[(1, 3), (2, 2), (2, 3), (3, 1), (3, 3)]


In [3]:
print("\nПример 3: m=3, n=4")
a3 = np.array([20, 30, 50])
b3 = np.array([25, 25, 25, 25])
c3 = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])

print("Поставки a:", a3)
print("Заявки b:", b3)
print("Матрица стоимостей c:")
print(c3)

status3, x3_opt, cost3, B3_opt, history3 = solve_transportation_problem(c3, a3, b3)

print(f"\nСтатус: {status3}")
print(f"Минимальная стоимость: {cost3}")
print(f"\nОптимальный план перевозок:")
print(x3_opt)
print(f"\nОптимальные базисные позиции (1-based):")
print(sorted([(i+1, j+1) for i, j in B3_opt]))


Пример 3: m=3, n=4
Поставки a: [20 30 50]
Заявки b: [25 25 25 25]
Матрица стоимостей c:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

Статус: optimal
Минимальная стоимость: 770.0

Оптимальный план перевозок:
[[20.  0.  0.  0.]
 [ 5. 25.  0.  0.]
 [ 0.  0. 25. 25.]]

Оптимальные базисные позиции (1-based):
[(1, 1), (2, 1), (2, 2), (3, 2), (3, 3), (3, 4)]
